In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
# sns.set_style('whitegrid')
import sys
import scipy
import os
import glob
import random

from scipy.interpolate import interp1d
import numpy.polynomial.polynomial as poly
import csv

from scipy.optimize import curve_fit
from sklearn import linear_model
from sklearn.preprocessing import MinMaxScaler

from datetime import datetime
import matplotlib.colors as colors2
from matplotlib import colormaps as cm
import matplotlib.cm as cmx

import pickle

#### The EoH performance data can be downloaded from the UK data service
The link is here: https://datacatalogue.ukdataservice.ac.uk/studies/study/9050#documentation

In [15]:
performanceSummaryPath = '../9050_EoH_working_folder/csv/cleaned_1/eoh_summary_for_publication.csv'
perfSummary_df = pd.read_csv(performanceSummaryPath)

In [16]:
includedIDs = perfSummary_df['Property_ID'].iloc[np.where(perfSummary_df['Included_SPF_analysis']==True)[0]].values
print('There are %d included IDs'%len(includedIDs))

There are 545 included IDs


In [17]:
perfSummary_df = perfSummary_df[perfSummary_df['Property_ID'].isin(includedIDs)]

#### The information about the installations is available
The link is here: https://usmart.io/org/esc/discovery/discovery-view-detail/d506e975-cb39-47d7-99e0-e99939a2c4bf

In [18]:
houseMetaPath = '../9050_EoH_working_folder/mrdoc/excel/BEIS Electrification of Heat Project - Property, Design and Installation Information.csv'
meta_df = pd.read_csv(houseMetaPath)

In [19]:
meta_df = meta_df[meta_df['Property_ID'].isin(includedIDs)]
print('we have the meta data for %d houses'%len(meta_df))

we have the meta data for 545 houses


In [25]:
includedIDs[0]

'EOH0003'

In [26]:
# get a list of files for all houses
fileNameList = []
for j in range(4):
    filePath = '../9050_EoH_working_folder/csv/cleaned_'+str(j+1)+'/P*'
    nameList = glob.glob(filePath)
    for entry in nameList:
        fileNameList.append(entry)

In [33]:
fileNameList[0].split('ID=')[1].split('.')[0]

['EOH0234', 'csv']

In [36]:
# create a dict of the fileName and the key (EOHID)
fileNameDict = {}
for entry in fileNameList:
    key = entry.split('ID=')[1].split('.')[0]
    fileNameDict[key] = entry
fileNameDict = {k: v for k, v in fileNameDict.items() if k in includedIDs}

In [48]:
houseSummary_df.head()

,Property_ID,Included_SPF_analysis,Excluded_SPF_analysis_reason,Whole_dataset_start,Whole_dataset_end,Whole_dataset_duration_days,Cleansed_dataset_start,Cleansed_dataset_end,Cleansed_dataset_duration_days,Selected_window_start,...,Coldest_day_IH_Energy_Consumed,Coldest_day_BUH_Energy_Consumed,Coldest_day_CP_Energy_Consumed,Coldest_day_Boiler_Energy_Output,CD_proportion_complete_Whole_System_Energy_Consumed,CD_proportion_complete_HP_Energy_Output,CD_proportion_complete_IH_Energy_Consumed,CD_proportion_complete_BUH_Energy_Consumed,CD_proportion_complete_CP_Energy_Consumed,CD_proportion_complete_Boiler_Energy_Output
1,EOH0003,True,NaN,15/04/2021 16:16,28/09/2023 23:58,896.32,15/04/2021 16:18,28/09/2023 23:56,896.32,23/05/2021,...,11.99,0.0,0.9,NaN,1.0,1.0,1.0,NaN,1.0,NaN


In [80]:
# for the first house, print some summary stats
EoH_ID = includedIDs[1]
# the filename convention is as follows:
houseSummary_df = perfSummary_df[perfSummary_df['Property_ID']==EoH_ID]
row = houseSummary_df.iloc[0]
count = 0
for col in houseSummary_df.columns:
    print(f"{col}\t{row[col]}")
    count+=1
    if count>15:
        break

Property_ID	EOH0005
Included_SPF_analysis	True
Excluded_SPF_analysis_reason	nan
Whole_dataset_start	21/05/2021 14:00
Whole_dataset_end	28/09/2023 23:58
Whole_dataset_duration_days	860.42
Cleansed_dataset_start	21/05/2021 14:02
Cleansed_dataset_end	28/09/2023 23:56
Cleansed_dataset_duration_days	860.41
Selected_window_start	27/07/2022
Selected_window_end	27/07/2023
Max_quality_score_selected_window	2.0
Mean_quality_score_selected_window	0.001
SPFH2_selected_window	3.476
SPFH3_selected_window	3.476
SPFH4_selected_window	3.405


In [81]:
fileName = fileNameDict[EoH_ID]
print(pd.to_datetime( datetime.strptime(houseSummary_df['Selected_window_start'].iloc[0], "%d/%m/%Y") ))
print(fileName)

2022-07-27 00:00:00
../9050_EoH_working_folder/csv/cleaned_1/Property_ID=EOH0005.csv


In [87]:
# let's just make sure that we can calculate the SPF correctly
df = pd.read_csv(fileName, header=0)

# keep only the data for the selected window
start_TS = pd.to_datetime( datetime.strptime(houseSummary_df['Selected_window_start'].iloc[0], "%d/%m/%Y") )
end_TS = pd.to_datetime( datetime.strptime(houseSummary_df['Selected_window_end'].iloc[0], "%d/%m/%Y") )

# convert the df column to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

mask = (df['Timestamp'] >= start_TS) & (df['Timestamp'] <= end_TS)
df = df.loc[mask,:]
df = df.set_index('Timestamp')

HP = df['Heat_Pump_Energy_Output'].iloc[-1] - df['Heat_Pump_Energy_Output'].iloc[0]
WS = df['Whole_System_Energy_Consumed'].iloc[-1] - df['Whole_System_Energy_Consumed'].iloc[0]
CP = df['Circulation_Pump_Energy_Consumed'].iloc[-1] - df['Circulation_Pump_Energy_Consumed'].iloc[0]

IH = (df['Immersion_Heater_Energy_Consumed'].iloc[-1] - df['Immersion_Heater_Energy_Consumed'].iloc[0]) if 'Immersion_Heater_Energy_Consumed' in df.columns else 0
BU = (df['Back-up_Heater_Energy_Consumed'].iloc[-1] - df['Back-up_Heater_Energy_Consumed'].iloc[0]) if 'Back-up_Heater_Energy_Consumed' in df.columns else 0

# SPF formulas (now unified for all cases)
SPF_H2 = HP / (WS - CP - IH - BU)
SPF_H3 = (HP + IH + BU) / (WS - CP)
SPF_H4 = (HP + IH + BU) / WS

print('SPF_H2 = %.2f' %SPF_H2)
print('SPF_H3 = %.2f' %SPF_H3)
print('SPF_H4 = %.2f' %SPF_H4)
print( 'The heat pump size is %.2f'%meta_df[meta_df['Property_ID']==EoH_ID]['HP_Size_kW'].values[0] )

SPF_H2 = 3.48
SPF_H3 = 3.48
SPF_H4 = 3.41
The heat pump size is 7.00


This is a test